## Ragas로 RAG 평가 이해하기
RAG 품질을 두 평가 영역으로 구분

- 검색(Retrieval): 필요한 문서를 찾았는가?
- 생성(Generation): 검색 문서에 근거해 질문에 답했는가?

예제 1건과 핵심 지표 4개를 사용한 입문 실습
Ragas 0.4.3 기준

Retrieval Augmented Generation Assessment (합성 테스트 데이터 셋)

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [3]:
from langchain_community.document_loaders import PDFPlumberLoader
import pdfplumber
loader = PDFPlumberLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]
len(docs)

19

In [4]:
docs[0].metadata

{'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [5]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

In [6]:
from ragas.testset import TestsetGenerator
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

generator_llm = ChatOpenAI(model="gpt-4o-mini")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#Ragas 0.4.x에서는 KeyphraseExtractor와 critic_llm을 직접 구성하지 않습니다.
generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings,
)

In [ ]:
from ragas.testset.transforms import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors import NERExtractor
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

transforms = [  # 페이지를 분할하지 않고 각 DOCUMENT에 필요한 정보를 추출
    
    SummaryExtractor(llm=generator.llm),    # LLM으로 문서 내용을 요약
    EmbeddingExtractor(    # 지정한 텍스트 속성을 숫자 벡터로 변환
        embedding_model= generator.embedding_model,
        property_name= "summary_embedding",
        embed_property_name= "summary",
    ),
    NERExtractor(llm=generator.llm),  # 개체명 인식(Named Entity Recognition)으로 사람·기관·장소 등 주요 개체를 추출
]
query_distribution = [                                            # 한 페이지의 개체 정보를 바탕으로 질문 생성
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm), 1.0),  # 하나의 문맥에서 답할 수 있는 구체적인 질문과 참조 정답을 생성
]

In [ ]:
testset = generator.generate_with_langchain_docs(
    documents= docs,   # PDF 문서에서 추출한 페이지를 DOCUMENT로 변환하여 전달
    testset_size= 10,  # 생성할 테스트셋의 질문 수
    transforms= transforms,  # 각 DOCUMENT에 적용할 변환기(요약, 임베딩, 개체명 인식)
    query_distribution= query_distribution,  # 한 페이지의 개체 정보를 바탕으로 질문 생성
    raise_exceptions= True,  # 질문 생성 중 오류 발생 시 예외를 발생시키도록 설정
)

Generating Samples: 100%|██████████| 10/10 [00:04<00:00,  2.19it/s]
